In [1]:
import json
import random
import pandas as pd
import numpy as np
import isodate
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- CONFIG ----------
PROC_FILE = "processed_videos_2.json"   # JSON or JSONL with full metadata
CLASS_FILE = "gpt-4o-mini_results.json"      # JSON or JSONL with fields: id, classification
# ----------------------------
def read_json_flexible(path):
    """
    Reads either JSONL (lines=True) or a normal JSON array.
    Returns a list of dicts.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")
    # try JSON lines
    with open(path, "r", encoding="utf-8") as f:
        first = f.readline()
        f.seek(0)
        try:
            # if first non-space char is '{' and file seems to have one json per line -> lines
            if first.lstrip().startswith("{"):
                # attempt to parse as JSONL
                items = []
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        items.append(json.loads(line))
                    except json.JSONDecodeError:
                        # fallback, try reading whole file
                        items = None
                        break
                if items is not None:
                    return items
            # else parse whole file as single JSON
            f.seek(0)
            data = json.load(f)
            if isinstance(data, list):
                return data
            elif isinstance(data, dict):
                # single dict -> wrap
                return [data]
            else:
                raise ValueError("Unknown JSON structure")
        except Exception:
            f.seek(0)
            data = json.load(f)
            if isinstance(data, list):
                return data
            elif isinstance(data, dict):
                return [data]
            else:
                raise

# ------------ load data ------------
proc_list = read_json_flexible(PROC_FILE)
class_list = read_json_flexible(CLASS_FILE)

# convert to DataFrame
df_proc = pd.DataFrame(proc_list)
df_class = pd.DataFrame(class_list)

# Normalize column names for ids
# proc: video_id ; class: id
if "video_id" not in df_proc.columns and "id" in df_proc.columns:
    df_proc = df_proc.rename(columns={"id": "video_id"})
if "id" not in df_class.columns and "video_id" in df_class.columns:
    df_class = df_class.rename(columns={"video_id": "id"})

# ---------- deduplicate processed videos by video_id ----------
before = len(df_proc)
df_proc = df_proc.drop_duplicates(subset=["video_id"], keep="first").reset_index(drop=True)
after = len(df_proc)
print(f"Processed videos: {before} rows -> {after} after dedup by video_id (removed {before-after})")

# ---------- merge classification into metadata ----------
# ensure df_class has 'id' and 'classification'
if "id" not in df_class.columns or "classification" not in df_class.columns:
    raise ValueError("classification file must contain 'id' and 'classification' fields")

df_merged = df_proc.merge(df_class[["id", "classification"]], left_on="video_id", right_on="id", how="left")
print(f"Merged dataset size: {len(df_merged)} (rows with classification: {df_merged['classification'].notna().sum()})")

# ---------- frequency of classes ----------
freq = df_merged["classification"].value_counts(dropna=False)
print("\nFrequência de classes (inclui NaN se houver vídeos sem classificação):")
print(freq)

freq_norm = df_merged['classification'].value_counts(normalize=True)
print(freq_norm)


# ---------- focus on relevant only ('sim') ----------
df_sim = df_merged[df_merged["classification"].str.strip().str.lower() == "sim"].copy()
print(f"\nVídeos marcados como 'sim': {len(df_sim)}")

# helper: parse duration to seconds safely
def parse_duration_to_seconds(d):
    try:
        if pd.isna(d):
            return np.nan
        if isinstance(d, (int, float)):
            return float(d)
        # often format like "PT2M29S"
        dur = isodate.parse_duration(d)
        return float(dur.total_seconds())
    except Exception:
        # fallback: try to extract digits (not recommended)
        return np.nan

# add numeric columns
for col in ["view_count", "like_count", "comment_count"]:
    if col in df_sim.columns:
        df_sim[col] = pd.to_numeric(df_sim[col], errors="coerce")
    else:
        df_sim[col] = np.nan

df_sim["duration_seconds"] = df_sim["duration"].apply(parse_duration_to_seconds)
stats = df_sim[["view_count", "like_count", "comment_count", "duration_seconds"]].describe()
print(stats)

print("\nMétricas dos vídeos classificados como 'sim':\n")
print(f"{'Métrica':<18} {'Média':<10} {'Mediana':<10} {'Desvio Padrão':<15} {'Coeficiente de Variação'}")
print("-" * 65)

for m, vals in stats.items():
    mean = vals['mean']
    median = vals['50%']
    std = vals['std']

    mean_str = f"{mean:.2f}" if mean is not None else "—"
    median_str = f"{median:.2f}" if median is not None else "—"
    std_str = f"{std:.2f}" if std is not None else "—"
    
    # Coeficiente de variação
    if mean is not None and mean != 0:
        cv = std / mean
        cv_str = f"{cv:.2f}"
    else:
        cv_str = "—"

    print(f"{m:<18} {mean_str:<10} {median_str:<10} {std_str:<15} {cv_str:<10}")




Processed videos: 21186 rows -> 21183 after dedup by video_id (removed 3)
Merged dataset size: 21183 (rows with classification: 21183)

Frequência de classes (inclui NaN se houver vídeos sem classificação):
classification
sim    14970
nao     6213
Name: count, dtype: int64
classification
sim    0.706699
nao    0.293301
Name: proportion, dtype: float64

Vídeos marcados como 'sim': 14970
         view_count    like_count  comment_count  duration_seconds
count  1.497000e+04  1.497000e+04   14970.000000      14970.000000
mean   2.388111e+04  5.556745e+02      16.860721        386.493253
std    5.398873e+05  1.319598e+04     220.621104       1169.781928
min    0.000000e+00  0.000000e+00       0.000000          0.000000
25%    3.200000e+01  1.000000e+00       0.000000         56.000000
50%    1.400000e+02  5.000000e+00       0.000000        115.000000
75%    7.820000e+02  2.500000e+01       2.000000        251.000000
max    4.453595e+07  1.288257e+06   18355.000000      43078.000000

Métrica

In [6]:
comments = pd.read_json('processed_videos_2.json', lines=True)
classif = pd.read_json('gpt-4o-mini_results.json', lines=True)
classif = classif.rename(columns={'id' : 'video_id'})
#classif = classif[classif['video_id'] != '2V5KZJgi_WA']
merged = comments.merge(classif[['video_id', 'classification']], on='video_id', how='left').drop_duplicates(subset=['video_id'], keep='first').reset_index(drop=True)
len(merged)

21183

In [7]:
merged_yes = merged[merged['classification'] == 'sim'].reset_index(drop=True)
len(merged_yes[merged_yes['comment_count'] >= 0])

14970